<div style="font-family: Arial, sans-serif; background-color: #f8f9fa; padding: 20px; border-radius: 10px;">
  <img src="https://sigmoidal.ai/wp-content/uploads/2024/09/Academia-Sigmoidal-Light.png" alt="Academia Sigmoidal" width="250">
  <h1 style="color: #007bff; font-size: 24px;">Bootcamp de Visão Computacional com Drones</h1>
  <h3 style="color: #343a40; font-size: 20px;">Projeto 4: Objetos pequenos e inferência em recortes</h3>
  <p style="color: #6c757d; font-size: 14px;"><strong>Instrutor:</strong> Carlos Melo, MSc.</p>
</div>


# Objetos pequenos e inferência em recortes

Vamos comparar imagem inteira e recortes sobrepostos usando exatamente o mesmo checkpoint. Avaliaremos detecção, contagem e tempo; os recortes podem melhorar, piorar ou apenas aumentar o custo.

Este notebook é independente: os dados necessários acompanham o repositório. No Colab, selecione **Copiar para o Drive** e execute as células na ordem. Selecione uma GPU em Ambiente de execução > Alterar tipo de ambiente de execução.

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/carlosfab/bootcamp-visao-computacional-drones/blob/codex/projeto-4-gado/projeto-4/03_objetos_pequenos.ipynb)


## Ambiente

A primeira célula obtém os arquivos da versão do projeto e instala as dependências. O PyTorch fornecido pelo ambiente será preservado; sua versão será registrada. Se já importou bibliotecas antes desta instalação, reinicie o ambiente e execute novamente desde o início.

Os arquivos de trabalho ficam em `dados/`, `pesos/` e `resultados/`. As pastas de métricas e figuras têm nomes fixos e seus arquivos são atualizados ao reexecutar. Antes de repetir o experimento ou encerrar o Colab, baixe o ZIP seguindo as instruções ao final.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
if not Path("suporte").is_dir():
    destino = Path("/content/bootcamp-gado")
    if not destino.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", "--branch", "codex/projeto-4-gado",
                        "https://github.com/carlosfab/bootcamp-visao-computacional-drones.git", str(destino)], check=True)
    subprocess.run(["git", "-C", str(destino), "sparse-checkout", "set", "projeto-4"], check=True)
    os.chdir(destino / "projeto-4")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


In [ ]:
import json
import hashlib
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from suporte.preparar_dados import obter_dados
from suporte.experimento import inventario, ler_caixas, desenhar
plt.rcParams.update({"figure.figsize": (9, 5), "font.size": 12,
                     "font.family": "sans-serif", "axes.grid": False})


A semente controla parte da aleatoriedade, mas versões, hardware e operações numéricas também influenciam o resultado. O registro abaixo identifica o ambiente desta execução, sem prometer identidade bit a bit entre CPU e GPU.


In [ ]:
import torch
from ultralytics import YOLO
from importlib.metadata import version
from suporte.experimento import inferir, calibrar, salvar_resultado
from suporte.contagem import evaluate_predictions
DISPOSITIVO = 0 if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DISPOSITIVO)


In [ ]:
SAIDA = Path("resultados/recortes")
SAIDA.mkdir(parents=True, exist_ok=True)
pacotes = ["numpy", "pandas", "matplotlib", "Pillow", "ultralytics", "torch"]
ambiente = {nome: version(nome) for nome in pacotes}
ambiente["python"] = platform.python_version()
ambiente["sistema"] = platform.platform()
ambiente["dispositivo"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
(SAIDA / "ambiente.json").write_text(json.dumps(ambiente, indent=2))
display(ambiente)


## Dados e partições

O recorte didático contém **300 imagens** de bovinos a pasto: 180 de treino, 60 de validação e 60 de teste. As imagens derivam do ICAERUS v2, com lado maior limitado a 2048 pixels e anotações YOLO de uma classe, `cow`. A atribuição e a licença CC BY 4.0 acompanham os dados.

Treino e validação usam Mauron e Jalogny em datas separadas. O teste usa Derval, uma fazenda ausente do desenvolvimento. Datas e voos não atravessam partições, mas os animais não têm identificadores individuais. A avaliação mede contagem **por imagem**, não um censo de indivíduos únicos.


In [ ]:
RAIZ = obter_dados()
tabela = inventario(RAIZ)
manifesto = json.loads((RAIZ / "manifesto.json").read_text())
treino = tabela[tabela["split"] == "train"].copy()
validacao = tabela[tabela["split"] == "val"].copy()
teste = tabela[tabela["split"] == "test"].copy()
assert [len(treino), len(validacao), len(teste)] == [180, 60, 60]
print("Imagens de treino, validação e teste:", len(treino), len(validacao), len(teste))


## Um checkpoint fixado

O pacote de pesos permite executar este notebook sem treinar antes. Os hashes verificam se usamos os mesmos bytes da distribuição, sem garantir qualidade preditiva. A avaliação será recalculada nesta sessão.

O projeto contém dois experimentos pareados pelas imagens de teste. No primeiro, baseline e fine-tuning mantêm arquitetura e inferência e mudam os pesos. Neste segundo, **imagem inteira e recortes mantêm exatamente o mesmo checkpoint de referência**. Não substitua a linha de imagem inteira deste notebook pelo resultado de seu treino anterior: seus hashes podem ser diferentes.

Para comparar seu próprio treinamento posteriormente, troque o checkpoint antes de calibrar e use outra pasta de resultados. Não misture pesos diferentes na comparação entre imagem inteira e recortes.


In [ ]:
import zipfile
from suporte.contagem import tile_windows, predict_image, nms
metadados_peso = json.loads(Path("assets/pesos-gado.json").read_text())
pacote_peso = Path("assets/pesos-gado.zip")
assert hashlib.sha256(pacote_peso.read_bytes()).hexdigest() == metadados_peso["archive_sha256"]
with zipfile.ZipFile(pacote_peso) as pacote:
    assert "pesos/gado.pt" in pacote.namelist()
    Path("pesos").mkdir(exist_ok=True)
    Path("pesos/gado.pt").write_bytes(pacote.read("pesos/gado.pt"))
PESO = Path("pesos/gado.pt")
assert hashlib.sha256(PESO.read_bytes()).hexdigest() == metadados_peso["weights_sha256"]
modelo = YOLO(str(PESO))


## Por que recortar?

Uma imagem com lado maior de 2048 pixels perde detalhe ao ser reduzida para uma entrada de 640. Um recorte de 640 pixels conserva mais pixels por animal, mas limita o contexto. A sobreposição reduz o risco de cortar um bovino na borda e também produz previsões repetidas.

O experimento fixa recortes de **640 pixels**, sobreposição nominal de **20%** e NMS global com **IoU de 0,5**. Nas bordas, a última janela é alinhada à imagem e a sobreposição real pode ser maior. Esses parâmetros são definidos antes de examinar o teste.


In [ ]:
pequeno = validacao[validacao["count"] > 0].sort_values("median_box_side").iloc[0]
foto = np.asarray(Image.open(pequeno["image"]).convert("RGB"))
janelas = tile_windows(foto.shape[1], foto.shape[0], tile_size=640, overlap=0.2)
print("Dimensões:", foto.shape)
print("Quantidade de recortes:", len(janelas))
print("Primeiras janelas:", janelas[:3])


In [ ]:
from matplotlib.patches import Rectangle
plt.figure(figsize=(12, 7))
plt.imshow(foto)
for x1, y1, x2, y2 in janelas:
    plt.gca().add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                                 fill=False, edgecolor="orange", linewidth=1))
plt.axis("off")
plt.show()


## Voltar ao referencial original

O detector devolve as caixas no referencial de cada recorte. Somar o deslocamento horizontal aos dois valores de `x` e o vertical aos dois valores de `y` recupera as coordenadas da imagem original. Só depois dessa transformação podemos comparar e deduplicar caixas de janelas diferentes.


In [ ]:
caixas, confiancas = [], []
for x1, y1, x2, y2 in janelas:
    previsao = predict_image(modelo, foto[y1:y2, x1:x2], conf=0.05, device=DISPOSITIVO)
    caixas.append(previsao["boxes"] + np.array([x1, y1, x1, y1]))
    confiancas.append(previsao["scores"])
caixas = np.concatenate(caixas)
confiancas = np.concatenate(confiancas)
mantidas = nms(caixas, confiancas, iou_threshold=0.5)
print("Caixas antes e depois da NMS:", len(caixas), len(mantidas))


A **supressão não máxima (NMS)** prioriza caixas de maior confiança e remove sobreposições acima do limiar. Ela reduz duplicatas, mas pode suprimir caixas de bovinos próximos ou manter fragmentos de um mesmo animal. Portanto, somar as contagens dos recortes é incorreto, e aplicar NMS não dispensa inspeção.

O limiar 0,25 abaixo é apenas uma escolha de exibição da demonstração na validação. O limiar de avaliação será calibrado usando todas as imagens de validação.


In [ ]:
gt = ler_caixas(pequeno["label"], foto.shape[1], foto.shape[0])
for titulo, indices in [("Antes da NMS", np.arange(len(caixas))), ("Depois da NMS", mantidas)]:
    print(titulo)
    figura = desenhar(foto, gt, caixas[indices], confiancas[indices], 0.25)
    plt.show()
    plt.close(figura)


## Comparação controlada na validação

A única mudança de inferência entre os métodos é processar a imagem inteira ou os recortes e reuni-los por NMS. Cada método tem seu limiar escolhido pela mesma regra de validação: maior F1, depois menor MAE e maior limiar.

Os recortes fazem várias chamadas ao detector por imagem. O tempo medido inclui inferência e pós-processamento, exclui leitura dos dados e pode incluir aquecimento. Execute ambos no mesmo dispositivo e evite interpretar pequenas diferenças como conclusivas.


In [ ]:
val_inteira = inferir(modelo, validacao, device=DISPOSITIVO, tiled=False)
limiar_inteira, curva_inteira = calibrar(val_inteira)
val_recortes = inferir(modelo, validacao, device=DISPOSITIVO, tiled=True)
limiar_recortes, curva_recortes = calibrar(val_recortes)
curva_inteira.to_csv(SAIDA / "calibracao_inteira.csv", index=False)
curva_recortes.to_csv(SAIDA / "calibracao_recortes.csv", index=False)
print("Limiar da imagem inteira:", limiar_inteira)
print("Limiar dos recortes:", limiar_recortes)


In [ ]:
salvar_resultado(SAIDA / "val_inteira", val_inteira, limiar_inteira)
salvar_resultado(SAIDA / "val_recortes", val_recortes, limiar_recortes)
configuracao = {"weights_sha256": metadados_peso["weights_sha256"], "imgsz": 640,
                "tile_size": 640, "overlap": 0.2, "global_nms_iou": 0.5,
                "conf_inteira": limiar_inteira, "conf_recortes": limiar_recortes,
                "iou_match": 0.5, "device": str(DISPOSITIVO),
                "manifest_sha256": hashlib.sha256((RAIZ / "manifesto.json").read_bytes()).hexdigest()}
(SAIDA / "configuracao.json").write_text(json.dumps(configuracao, indent=2))
display(configuracao)


## Avaliar o teste com as decisões congeladas

Agora aplicaremos os dois métodos às mesmas imagens de Derval. A tabela deve sustentar uma decisão entre benefício e custo, inclusive quando os recortes não ajudam. Nenhuma mudança de parâmetros será feita a partir deste resultado.


In [ ]:
teste_inteira = inferir(modelo, teste, device=DISPOSITIVO, tiled=False)
teste_recortes = inferir(modelo, teste, device=DISPOSITIVO, tiled=True)
resumo_inteira = salvar_resultado(SAIDA / "test_inteira", teste_inteira, limiar_inteira)
resumo_recortes = salvar_resultado(SAIDA / "test_recortes", teste_recortes, limiar_recortes)
chaves = ["n_images", "precision", "recall", "f1", "mae", "rmse", "bias", "exact_accuracy"]
comparacao = {nome: {chave: resumo[chave] for chave in chaves}
             for nome, resumo in [("inteira", resumo_inteira), ("recortes", resumo_recortes)]}
for nome, resumo in [("inteira", resumo_inteira), ("recortes", resumo_recortes)]:
    comparacao[nome]["segundos_por_imagem"] = resumo["timing"]["mean_seconds"]
comparacao = pd.DataFrame.from_dict(comparacao, orient="index")
comparacao.to_csv(SAIDA / "comparacao.csv")
display(comparacao)


O custo depende da quantidade de janelas. Abaixo registramos essa quantidade para cada imagem e mostramos separadamente positivos e negativos. Taxa alta de contagem exata em cenas vazias não basta para concluir que o detector conta bem as cenas com gado.


In [ ]:
custo = teste[["image_id", "width", "height"]].copy()
custo["recortes"] = [len(tile_windows(int(w), int(h)))
                     for w, h in zip(custo["width"], custo["height"])]
custo.to_csv(SAIDA / "janelas_por_imagem.csv", index=False)
display(custo["recortes"].describe())
display({"inteira": resumo_inteira["by_presence"], "recortes": resumo_recortes["by_presence"]})


## Examinar erros concretos

A galeria seleciona as imagens com mais falsos positivos e falsos negativos, usando o limiar já congelado. **Verde** representa a anotação; **vermelho tracejado**, a previsão. Uma contagem correta pode esconder uma omissão compensada por uma caixa falsa.

Depois desta inspeção, registre hipóteses de falha. Uma alteração motivada pelo teste passa a ser exploração e exigiria outro conjunto independente para uma nova avaliação final.


In [ ]:
por_imagem = pd.read_csv(SAIDA / "test_recortes" / "per_image.csv")
por_imagem["falhas"] = por_imagem["fp"] + por_imagem["fn"]
casos = por_imagem.sort_values(["falhas", "image_id"], ascending=[False, True]).head(4)
display(casos[["image_id", "gt_count", "pred_count", "fp", "fn"]])
indice = {linha["image_id"]: linha for linha in teste_recortes}
(SAIDA / "galeria").mkdir(exist_ok=True)


In [ ]:
for numero, image_id in enumerate(casos["image_id"]):
    linha = indice[image_id]
    figura = desenhar(linha["image"], linha["gt_boxes"], linha["pred_boxes"],
                      linha["scores"], limiar_recortes)
    figura.savefig(SAIDA / "galeria" / f"caso_{numero}.png", dpi=120)
    plt.show()
    plt.close(figura)


Compare as mesmas imagens selecionadas acima com a previsão na imagem inteira. Observe animais recuperados, caixas falsas e possíveis duplicatas ou supressões. Não pressuponha que todas essas falhas ocorrerão neste conjunto.


In [ ]:
indice_inteira = {linha["image_id"]: linha for linha in teste_inteira}
for image_id in casos["image_id"]:
    linha = indice_inteira[image_id]
    figura = desenhar(linha["image"], linha["gt_boxes"], linha["pred_boxes"],
                      linha["scores"], limiar_inteira)
    plt.show()
    plt.close(figura)


## Uma recomendação baseada em evidências

Recomende uma configuração para o cenário avaliado. Use MAE, viés, F1, erros em positivos e negativos, custo e pelo menos quatro exemplos para justificar. Identifique o hash dos pesos em cada linha da tabela final. O efeito dos recortes é medido pelo par deste notebook, com o mesmo checkpoint; diferenças para o treino do notebook anterior não podem ser atribuídas só ao tiling. Distinga o que os resultados demonstram do que permanece uma hipótese.

Este experimento não resolve identidade entre fotografias, contagem de um rebanho inteiro ou funcionamento em qualquer fazenda. A próxima avaliação poderia reservar outros voos e fazendas, com referência independente e um protocolo de cobertura definido.


## Salvar a execução

A pasta de um treinamento concluído é protegida contra reutilização; uma nova tentativa exige outro `name`. Já as pastas de métricas, configurações e figuras têm nomes fixos e são atualizadas ao reexecutar os notebooks. Baixe os resultados **antes de repetir** ou encerrar o runtime.

No Colab, copie o trecho opcional abaixo para uma nova célula e execute-o quando quiser baixar a execução. O ZIP inclui `resultados/` inteiro, os pesos treinados que estiverem nessa pasta e o manifesto. O checkpoint de referência do notebook de recortes continua disponível nos arquivos do projeto e é identificado pelo hash da configuração.

```python
import shutil
from datetime import datetime
from google.colab import files
shutil.copy("assets/manifesto.json", "resultados/manifesto.json")
nome = "resultados-projeto4-" + datetime.now().strftime("%Y%m%d-%H%M%S")
arquivo = shutil.make_archive(nome, "zip", "resultados")
files.download(arquivo)
```

Salve também este notebook com suas saídas em **Arquivo > Salvar** na cópia do Drive, ou use **Arquivo > Fazer download > Fazer download do .ipynb**. A cópia do notebook não salva automaticamente os arquivos temporários do runtime. Localmente, compacte `resultados/` ou use as mesmas linhas sem importar `google.colab` e sem chamar `files.download`.
